## Preparando os dados via API
**ccxt:** CryptoCurrency eXchange Trading - conectar e negociar em diversas exchanges de criptomoedas

**Alguns significados**
- O (Open / Abertura): O preço do ativo no exato momento em que o intervalo de tempo começou (ex: o preço às 10:00:00)
- H (High / Máxima): O preço mais alto que o ativo atingiu durante aquele intervalo
- L (Low / Mínima): O preço mais baixo que o ativo atingiu durante aquele intervalo
- C (Close / Fechamento): O preço do ativo no momento em que o intervalo terminou (ex: o preço às 10:59:59)
- V (Volume): A quantidade total do ativo que foi negociada (comprada/vendida) naquele período

In [0]:
!pip install ccxt

In [0]:
import json
import os

# configurações do projeto
config_path = os.path.join(os.path.dirname(os.getcwd()), 'config', 'trading_params.json')
with open(config_path, 'r') as f:
    config = json.load(f)

# extrai configurações
PAIRS = config['symbols']['active']
TIMEFRAMES = [config['data']['main_timeframe']] + config['data']['auxiliary_timeframes']
# remove duplicatas e ordena
TIMEFRAMES = sorted(list(set(TIMEFRAMES)))
START_DATE = config['data']['start_date']
MAX_CANDLES_TARGET = 50000

print(f"\nSímbolos ativos ({len(PAIRS)})")
for pair in PAIRS:
    print(f"   • {pair}")
print(f"\nTimeframes: {TIMEFRAMES}")
print(f"Data inicial: {START_DATE}")

In [0]:
%skip
symbols = list(binance.load_markets().keys())
df_symbols = spark.createDataFrame([(s,) for s in symbols], ["symbol"])
df_symbols.display()

In [0]:
import ccxt
import time
from typing import List, Dict, Tuple
import pyspark.sql.types as t
import pyspark.sql.functions as f
from datetime import datetime, timedelta

# configuração da API
binance = ccxt.binanceus({
    'enableRateLimit': True,
    'timeout': 30000,  # 30s timeout
    'options': {'defaultType': 'spot'}
})

print(f"  • Pares: {len(PAIRS)}")
print(f"  • Timeframes: {TIMEFRAMES}")
print(f"  • Início: {START_DATE}")
print(f"  • Exchange: {binance}")

In [0]:
def fetch_ohlcv_robust(
    exchange: ccxt.Exchange,
    symbol: str,
    timeframe: str,
    start_date: str,
    max_candles: int = 50000,
    batch_size: int = 1000,
    max_retries: int = 5
) -> List[List]:
    """
    Coleta dados OHLCV com retry logic robusto e tratamento de erros
    
    Args:
        exchange: Instância ccxt da exchange
        symbol: Par de trading (ex: 'BTC/USDT')
        timeframe: Intervalo de tempo ('15m', '1h', '4h')
        start_date: Data inicial ISO 8601 format
        max_candles: Número máximo de candles a coletar
        batch_size: Tamanho do lote por requisição
        max_retries: Número máximo de tentativas por requisição
    
    Returns:
        Lista de candles OHLCV
    """
    all_candles = []
    since = exchange.parse8601(start_date)
    
    print(f"  Coletando {symbol} [{timeframe}]...", end="")
    
    while len(all_candles) < max_candles:
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            try:
                # tenta buscar o lote
                new_candles = exchange.fetch_ohlcv(
                    symbol, 
                    timeframe=timeframe, 
                    since=since, 
                    limit=batch_size
                )
                
                # se não retornou nada, chegamos ao fim
                if not new_candles:
                    print(f" Fim do histórico ({len(all_candles)} candles)")
                    return all_candles
                
                # adiciona novos candles
                all_candles.extend(new_candles)
                
                # atualiza timestamp para próxima iteração
                since = new_candles[-1][0] + 1
                
                # pausa para respeitar rate limit
                time.sleep(exchange.rateLimit / 1000)
                
                success = True
                
                # Feedback de progresso a cada 5000 candles
                if len(all_candles) % 5000 == 0:
                    print(f" {len(all_candles)}...", end="")
                
            except ccxt.NetworkError as e:
                retry_count += 1
                wait_time = 2 ** retry_count  # Exponential backoff
                print(f" Network error (retry {retry_count}/{max_retries})", end="")
                time.sleep(wait_time)
                
            except ccxt.ExchangeError as e:
                retry_count += 1
                wait_time = 2 ** retry_count
                print(f" Exchange error (retry {retry_count}/{max_retries})", end="")
                time.sleep(wait_time)
                
            except Exception as e:
                print(f" Erro inesperado: {str(e)[:100]}")
                return all_candles  # Retorna o que conseguimos coletar
        
        # se esgotou todas as tentativas
        if not success:
            print(f" Falhou após {max_retries} tentativas")
            return all_candles
    
    print(f" Completo ({len(all_candles)} candles)")
    return all_candles

In [0]:
from datetime import datetime

# Dicionário para armazenar todos os dados coletados
# Estrutura: collected_data[timeframe][symbol] = list_of_candles
collected_data = {tf: {} for tf in TIMEFRAMES}

start_time = datetime.now()
print(f"\nIniciando coleta em larga escala...")
print(f"   Hora de início: {start_time.strftime('%H:%M:%S')}\n")

total_pairs = len(PAIRS) * len(TIMEFRAMES)
current = 0

for timeframe in TIMEFRAMES:
    print(f"\nTimeframe: {timeframe}")
    print("=" * 60)
    
    for symbol in PAIRS:
        current += 1
        print(f"[{current}/{total_pairs}] ", end="")
        
        try:
            candles = fetch_ohlcv_robust(
                exchange=binance,
                symbol=symbol,
                timeframe=timeframe,
                start_date=START_DATE,
                max_candles=MAX_CANDLES_TARGET
            )
            
            collected_data[timeframe][symbol] = candles
            
        except Exception as e:
            print(f"  ERRO crítico em {symbol}: {str(e)[:100]}")
            collected_data[timeframe][symbol] = []  # lista vazia para não quebrar pipeline

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n{'='*60}")
print(f"Coleta concluída em {duration:.1f}s ({duration/60:.1f} min)")
print(f"\nResumo da coleta:")

total_candles = 0
for tf in TIMEFRAMES:
    tf_total = sum(len(candles) for candles in collected_data[tf].values())
    total_candles += tf_total
    print(f"  • {tf}: {tf_total:,} candles")

print(f"\nTotal geral: {total_candles:,} candles")

In [0]:
# schema para OHLCV
schema = t.StructType([
    t.StructField("timestamp", t.LongType(), True),
    t.StructField("open", t.DoubleType(), True),
    t.StructField("high", t.DoubleType(), True),
    t.StructField("low", t.DoubleType(), True),
    t.StructField("close", t.DoubleType(), True),
    t.StructField("volume", t.DoubleType(), True)
])

# Dicionário para armazenar DataFrames
# Estrutura: spark_dfs[timeframe][symbol] = DataFrame
spark_dfs = {tf: {} for tf in TIMEFRAMES}

for timeframe in TIMEFRAMES:
    print(f"{timeframe}:")
    
    for symbol in PAIRS:
        candles = collected_data[timeframe][symbol]
        
        if len(candles) > 0:
            # Cria DataFrame
            df = spark.createDataFrame(candles, schema)
            
            # Converte timestamp de milissegundos para timestamp SQL
            df = df.withColumn(
                "timestamp", 
                (f.col("timestamp") / 1000).cast("timestamp")
            )
            
            # Converte para fuso horário de São Paulo
            df = df.withColumn(
                "timestamp", 
                f.from_utc_timestamp("timestamp", "America/Sao_Paulo")
            )
            
            # Adiciona metadados
            df = df.withColumn("symbol", f.lit(symbol))
            df = df.withColumn("timeframe", f.lit(timeframe))
            
            # Ordena por timestamp
            df = df.orderBy("timestamp")
            
            spark_dfs[timeframe][symbol] = df
            
            print(f"  {symbol}: {df.count():,} registros")
        else:
            print(f"  {symbol}: sem dados")

## Consolidação e Validação dos Dados

Agora vamos:
1. Consolidar todos os pares em tabelas únicas por timeframe
2. Validar a qualidade dos dados (nulos, duplicatas, gaps)
3. Salvar em Delta Lake para uso no modelo

In [0]:
# dicionário para armazenar as tabelas consolidadas
consolidated_tables = {}

for timeframe in TIMEFRAMES:
    print(f"Consolidando {timeframe}:")
    
    # Lista de DataFrames para unir
    dfs_to_union = []
    
    for symbol in PAIRS:
        if symbol in spark_dfs[timeframe]:
            df = spark_dfs[timeframe][symbol]
            
            # Adiciona apenas se tiver dados
            if df.count() > 0:
                dfs_to_union.append(df)
    
    # Faz union de todos os DataFrames
    if len(dfs_to_union) > 0:
        consolidated_df = dfs_to_union[0]
        
        for df in dfs_to_union[1:]:
            consolidated_df = consolidated_df.union(df)
        
        # Ordena por symbol e timestamp
        consolidated_df = consolidated_df.orderBy("symbol", "timestamp")
        
        # Remove duplicatas (caso existam)
        consolidated_df = consolidated_df.dropDuplicates(["symbol", "timestamp"])
        
        consolidated_tables[timeframe] = consolidated_df
        
        # Estatísticas
        total_rows = consolidated_df.count()
        symbol_counts = consolidated_df.groupBy("symbol").count().orderBy("symbol")
        
        print(f"  Total: {total_rows:,} registros")
        print(f"  • Distribuição por par:")
        for row in symbol_counts.collect():
            print(f"     - {row['symbol']}: {row['count']:,}")
    else:
        print(f"  Nenhum dado para consolidar")
    
    print()

In [0]:
for timeframe in TIMEFRAMES:
    if timeframe not in consolidated_tables:
        continue
    
    df = consolidated_tables[timeframe]
    
    print(f"{timeframe}:")
    print("=" * 60)
    
    # Verifica nulos
    null_counts = df.select(
        [
            f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in ["open", "high", "low", "close", "volume"]
        ]
    ).collect()[0].asDict()
    
    has_nulls = any(count > 0 for count in null_counts.values())
    
    if has_nulls:
        print(f"  Nulos encontrados:")
        for col, count in null_counts.items():
            if count > 0:
                print(f"     - {col}: {count}")
    else:
        print(f"  Nenhum valor nulo")
    
    # Verifica range de datas
    date_stats = df.agg(
        f.min("timestamp").alias("min_date"),
        f.max("timestamp").alias("max_date")
    ).collect()[0]
    
    print(f"  • Período: {date_stats['min_date']} a {date_stats['max_date']}")
    
    # Verifica valores negativos ou zeros
    invalid_prices = df.filter(
        (f.col("open") <= 0) | 
        (f.col("high") <= 0) | 
        (f.col("low") <= 0) | 
        (f.col("close") <= 0)
    ).count()
    
    if invalid_prices > 0:
        print(f"  Preços inválidos (≤ 0): {invalid_prices}")
    else:
        print(f"  Todos os preços válidos (> 0)")
    
    # Verifica consistência OHLC (high >= low, etc)
    inconsistent_ohlc = df.filter(
        (f.col("high") < f.col("low")) |
        (f.col("high") < f.col("open")) |
        (f.col("high") < f.col("close")) |
        (f.col("low") > f.col("open")) |
        (f.col("low") > f.col("close"))
    ).count()
    
    if inconsistent_ohlc > 0:
        print(f"  Candles inconsistentes: {inconsistent_ohlc}")
    else:
        print(f"  OHLC consistente")
    
    print()

print("Validação concluída")

In [0]:
# Mostra uma amostra de dados para cada par (timeframe 1h)
if '1h' in consolidated_tables:
    print("\nAmostra dos dados (timeframe 1h):\n")
    
    df_1h = consolidated_tables['1h']
    
    # Pega últimos 5 registros de cada par
    from pyspark.sql.window import Window
    
    w = Window.partitionBy("symbol").orderBy(f.col("timestamp").desc())
    
    sample_df = (
        df_1h
        .withColumn("row_num", f.row_number().over(w))
        .filter(f.col("row_num") <= 5)
        .drop("row_num")
        .orderBy("symbol", f.col("timestamp").desc())
    )
    
    display(sample_df)

In [0]:
for timeframe in TIMEFRAMES:
    if timeframe not in consolidated_tables:
        continue
    
    df = consolidated_tables[timeframe]
    table_name = f"default.crypto_ohlcv_{timeframe.replace('m', 'min').replace('h', 'hour')}"
    
    print(f"  • Salvando {table_name}...", end=" ")
    
    try:
        (
            df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("mergeSchema", "true")
            .partitionBy("symbol")  # Particiona por symbol para queries eficientes
            .saveAsTable(table_name)
        )
        
        print(f"{df.count():,} registros salvos")
        
    except Exception as e:
        print(f"Erro: {str(e)[:100]}")

print("\nTabelas criadas:")
print("  • default.crypto_ohlcv_15min  (15 minutos)")
print("  • default.crypto_ohlcv_1hour   (1 hora)")
print("  • default.crypto_ohlcv_4hour   (4 horas)")

## Próximos passos

1. **Feature Engineering Multi-Timeframe:**
   - Agregar features de 15m, 1h e 4h para capturar diferentes escalas temporais
   - Criar features de correlação entre pares
   - Adicionar indicadores técnicos avançados

2. **Features de Sentiment (se APIs disponíveis):**
   - Funding rate
   - Open interest
   - Long/short ratio
   - Volume profile

3. **Modelagem Melhorada:**
   - Target adaptativo baseado em ATR
   - Balanceamento de classes mais sofisticado
   - Walk-forward validation